# YaSpeech: автопротоколирование деловых встреч на Yandex Cloud
### SpeechKit (ASR) + YandexGPT + Object Storage

Кукбук показывает, как собрать сервис **«запись встречи → готовый протокол»** на трёх сервисах Yandex Cloud.


# 1. Введение

## Назначение

В этом кукбуке демонстрируется, как собрать сервис **«запись встречи → готовый протокол»** с помощью:
- **SpeechKit STT v3** — асинхронное распознавание речи с разметкой по каналам
- **YandexGPT** — диаризация по составу команды, коррекция ошибок ASR, генерация протокола
- **Object Storage** — промежуточное хранение аудио (обязательно: SpeechKit async читает файл только по ссылке)

## Описание задачи

**Бизнес-задача**: команды, которые фиксируют решения и задачи по итогам планёрок, тратят время на ручное протоколирование. Этот кукбук демонстрирует:

- Распознавание речи из аудиозаписи встречи
- Коррекцию типичных ошибок ASR (разорванные слова, аббревиатуры, произнесённые словами числа)
- Генерацию структурированного протокола: решения, задачи с ответственными, открытые вопросы

**Основные сервисы Yandex Cloud:**
- SpeechKit STT v3
- AI Studio с YandexGPT
- Object Storage

***

## Ожидаемый результат

Готовый пайплайн «аудио → протокол» — рабочий каркас, от которого можно оттолкнуться и построить решение под свою задачу.


# 2. Архитектура решения

## Компоненты системы

```
Аудио (.wav/.ogg/.mp3)
      │
      ▼
Object Storage (SpeechKit читает аудио только по ссылке)
      │
      ▼
SpeechKit STT v3 — асинхронное распознавание
      │
      ▼
Один вызов YandexGPT:
  диаризация по составу команды + коррекция ASR + протокол
      │
      ▼
Структурированный протокол (JSON)
```

| Компонент | Назначение | Роли и права |
|---|---|---|
| **SpeechKit STT v3** | Распознавание речи (`recognizeFileAsync` + `getRecognition`) | `ai.speechkit-stt.user` — вызов распознавания |
| **YandexGPT** | Один вызов: разметка спикеров по составу команды, коррекция ошибок ASR, протокол | `ai.languageModels.user` — генерация текста |
| **Object Storage** | Промежуточное хранение аудио для SpeechKit | `storage.editor` — создание бакета и загрузка объектов |

## Описание ролей и их области действия

**Сервисный аккаунт** — учётная запись, от имени которой приложения и автоматизированные сервисы обращаются к ресурсам Yandex Cloud. В отличие от обычных пользовательских аккаунтов, сервисные аккаунты используются для программного доступа и не требуют браузерной аутентификации.

- `ai.speechkit-stt.user` — *область действия: каталог и вложенные ресурсы.* Позволяет отправлять запросы на распознавание речи в SpeechKit (`recognizeFileAsync`, `getRecognition`).
- `ai.languageModels.user` — *область действия: каталог и вложенные ресурсы.* Минимальная роль для работы с моделями генерации текста YandexGPT — отправка запросов на генерацию.
- `storage.editor` — *область действия: каталог, вложенные ресурсы, либо конкретный бакет.* Даёт право создавать бакет и загружать в него объекты — обе операции нужны коду кукбука (`ensure_bucket`, `upload_audio`). Более узкая роль `storage.uploader` разрешает только загрузку в уже существующий бакет, но не его создание — если бакет уже создан заранее, можно использовать её вместо `storage.editor`.

## Как назначить роль сервисному аккаунту через консоль Yandex Cloud

1. Откройте сервис Identity and Access Management (IAM) в консоли управления.
2. Выберите каталог, к которому нужно предоставить доступ.
3. Перейдите на вкладку «Права доступа» → «Настроить доступ».
4. Выберите раздел «Сервисные аккаунты», найдите нужный или создайте новый.
5. Нажмите «Добавить роль» и выберите роли из списка выше — можно назначить несколько ролей сразу.
6. Сохраните изменения.


# 3. Подготовка окружения

Понадобятся три вещи:
- **FOLDER_ID** — идентификатор каталога: https://yandex.cloud/ru/docs/resource-manager/operations/folder/get-id
- **YC_API_KEY** — ключ сервисного аккаунта с ролями `ai.speechkit-stt.user`, `ai.languageModels.user`, `storage.editor` (см. раздел 2): https://yandex.cloud/ru/docs/iam/operations/api-key/create
- **AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY** — статические ключи для Object Storage: https://yandex.cloud/ru/docs/iam/concepts/authorization/access-key


Устанавливаем нужные библиотеки.


In [ ]:
!pip install -q openai python-dotenv boto3 requests pydantic


In [ ]:
import os
import json
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import boto3
import requests
import openai
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field, field_validator

load_dotenv(find_dotenv())

FOLDER_ID = os.getenv("FOLDER_ID", "YOUR_FOLDER_ID")
YC_API_KEY = os.getenv("YC_API_KEY", "YOUR_API_KEY")
GPT_MODEL = os.getenv("GPT_MODEL", "yandexgpt-5-lite")  # см. актуальный список моделей: https://yandex.cloud/ru/docs/ai-studio/concepts/generation/models
MODEL_URI = f"gpt://{FOLDER_ID}/{GPT_MODEL}"

gpt_client = openai.OpenAI(
    api_key=YC_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
)

S3_BUCKET = os.getenv("BUCKET_NAME", "YOUR_BUCKET_NAME")
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "YOUR_AWS_ACCESS_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "YOUR_AWS_SECRET_KEY"),
    region_name="ru-central1",
)

SPEECHKIT_RECOGNIZE_URL = "https://stt.api.cloud.yandex.net/stt/v3/recognizeFileAsync"
SPEECHKIT_GET_URL = "https://stt.api.cloud.yandex.net/stt/v3/getRecognition"
OPERATION_URL = "https://operation.api.cloud.yandex.net/operations"

print("Клиенты инициализированы:", MODEL_URI)


# 4. Загрузка аудио в Object Storage

SpeechKit async принимает аудио **только по ссылке** на Object Storage — поэтому bucket и upload обязательны, даже в учебном примере.

Поддерживаемые контейнеры: **WAV, OggOpus, MP3** (без M4A).


In [ ]:
def ensure_bucket(bucket_name: str) -> None:
    """Проверяет, есть ли уже такой бакет, и создаёт его, если нет."""
    existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
    if bucket_name not in existing:
        s3.create_bucket(Bucket=bucket_name)
    print(f"Бакет готов: {bucket_name}")


def upload_audio(local_path: str, key: str) -> str:
    """Загружает локальный аудиофайл в Object Storage и возвращает
    публичную ссылку на него — эту ссылку потом передаём в SpeechKit."""
    s3.upload_file(local_path, S3_BUCKET, key)
    uri = f"https://storage.yandexcloud.net/{S3_BUCKET}/{key}"
    print(f"Загружено: {uri}")
    return uri


ensure_bucket(S3_BUCKET)


# 5. SpeechKit STT v3: асинхронное распознавание

Отправляем аудио на распознавание — SpeechKit сразу отвечает номером операции и продолжает работу в фоне. Мы периодически спрашиваем «готово?», а когда готово — забираем результат: текст, разбитый на реплики.

У SpeechKit есть встроенная функция `speakerLabeling` — она пытается сама определить, кто где говорит, но работает только для одноканальной записи и максимум на двух дикторов, и на практике часто ошибается: если собеседники говорят в один микрофон, границы между репликами получаются смазанными. Это скорее разметка по акустике и паузам, чем настоящая диаризация — она не понимает, кто есть кто, только угадывает смену голоса. Поэтому мы отключаем эту функцию у SpeechKit и определяем спикеров сами — на основе текста: передаём весь транскрипт в LLM, и она сама разбивает его на реплики и подставляет реальные имена участников по составу команды.

Скорость примерно такая: минута записи распознаётся около 10 секунд ([подробнее в документации](https://aistudio.yandex.ru/docs/ru/speechkit/stt/api/transcribation-api-v3)).


In [ ]:
def start_recognition(audio_uri: str, language: str = "ru-RU") -> str:
    """Запускает асинхронное распознавание по ссылке на файл в Object Storage
    и сразу возвращает id операции — сам результат появится позже."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}", "Content-Type": "application/json"}
    body = {
        "uri": audio_uri,
        "recognitionModel": {
            "model": "general",
            "audioFormat": {"containerAudio": {"containerAudioType": "WAV"}},
            "textNormalization": {
                "textNormalization": "TEXT_NORMALIZATION_ENABLED",
                "profanityFilter": False,
                "literatureText": True,
            },
            "languageRestriction": {"restrictionType": "WHITELIST", "languageCode": [language]},
        },
        # speakerLabeling нарочно не включаем — она угадывает смену голоса по
        # акустике, а не понимает, кто есть кто; спикеров определяем сами по
        # тексту через LLM (см. раздел 6).
    }
    resp = requests.post(SPEECHKIT_RECOGNIZE_URL, headers=headers, json=body, timeout=30)
    resp.raise_for_status()
    operation_id = resp.json()["id"]
    print(f"Операция запущена: {operation_id}")
    return operation_id


def wait_operation(operation_id: str, poll_interval_s: int = 5, max_wait_s: int = 900) -> None:
    """Опрашивает статус операции распознавания, пока она не завершится
    (done: true), или бросает исключение по таймауту/ошибке SpeechKit."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    waited = 0
    while waited < max_wait_s:
        resp = requests.get(f"{OPERATION_URL}/{operation_id}", headers=headers, timeout=15)
        resp.raise_for_status()
        op = resp.json()
        if op.get("done"):
            if op.get("error"):
                raise RuntimeError(f"SpeechKit error: {op['error']}")
            return
        time.sleep(poll_interval_s)
        waited += poll_interval_s
    raise TimeoutError("SpeechKit: не завершилось за отведённое время")


def fetch_recognition(operation_id: str) -> List[Dict[str, Any]]:
    """Забирает результат завершённой операции — ответ приходит построчным
    JSON (NDJSON), функция разбирает его в список {channelTag, text}."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    resp = requests.get(SPEECHKIT_GET_URL, headers=headers,
                        params={"operationId": operation_id}, timeout=60)
    resp.raise_for_status()

    chunks = []
    for line in resp.text.splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        final = obj.get("result", {}).get("final")
        if not final:
            continue
        alternatives = final.get("alternatives", [])
        if not alternatives or not alternatives[0].get("text", "").strip():
            continue
        chunks.append({"channelTag": final.get("channelTag", "0"), "text": alternatives[0]["text"].strip()})
    return chunks


def recognize_meeting_audio(audio_uri: str) -> List[Dict[str, Any]]:
    """Полный цикл распознавания одного файла: запустить, дождаться,
    забрать результат."""
    operation_id = start_recognition(audio_uri)
    wait_operation(operation_id)
    return fetch_recognition(operation_id)


## 5.1 Постобработка транскрипта

Склеиваем подряд идущие реплики одного канала и фильтруем короткие слова-паразиты.


In [ ]:
NOISE_MARKERS = {"угу", "ага", "эм", "эээ", "ну", "вот", "это", "так"}


def is_noise_segment(text: str) -> bool:
    """Короткая реплика из одних слов-паразитов («угу», «ну» и т.п.) —
    такие сегменты не несут смысла, их отфильтровываем."""
    words = text.lower().split()
    return bool(words) and len(words) <= 3 and all(w in NOISE_MARKERS for w in words)


def postprocess_transcript(chunks: List[Dict[str, Any]]) -> str:
    """Склеивает реплики одного канала подряд, фильтрует мусор, отдаёт
    один текст с метками [Канал N] — окончательную разметку спикеров
    (диаризацию по именам) делает LLM-проход ниже."""
    filtered = [c for c in chunks if c["text"] and not is_noise_segment(c["text"])]

    merged: List[Dict[str, str]] = []
    for c in filtered:
        if merged and merged[-1]["channel"] == c["channelTag"]:
            merged[-1]["text"] += " " + c["text"]
        else:
            merged.append({"channel": c["channelTag"], "text": c["text"]})

    return "\n".join(f"[Канал {m['channel']}] {m['text']}" for m in merged)


# 6. YandexGPT: спикеры + коррекция + протокол

Для коротких встреч — один структурированный запрос: модель получает сырой транскрипт (с грубой разметкой по каналам) и состав команды проекта, а возвращает исправленный текст с реальными именами и готовый протокол.

Три инструкции, которые здесь особенно важны:
- **не выдумывай** — пустой список лучше вымышленного решения или задачи;
- **не меняй числа, даты и суммы** при исправлении ошибок распознавания;
- **разметка по каналам — подсказка, не факт** — модель может перегруппировать реплики, если разметка SpeechKit похожа на шум.

Длинный транскрипт целиком в один ответ не поместится: `max_output_tokens` ограничивает объём генерации, а ответ должен содержать исправленный текст целиком — при обрыве генерация просто останавливается на середине JSON, и такой ответ невозможно разобрать. Поэтому если текст длиннее порога, ниже он режется на чанки по репликам — каждый чанк исправляется отдельным вызовом (без протокола, это дешевле по токенам), исправленные куски склеиваются, и уже по всему тексту отдельным вызовом собирается протокол.


In [ ]:
class ActionItem(BaseModel):
    owner: str = Field(description="Ответственный, или 'Команда' если не назван")
    task: str
    deadline: Optional[str] = None


class ProtocolOutput(BaseModel):
    """Pydantic-модель не диктует формат ответа модели (Responses API это
    делает текстовым промптом, см. build_protocol_prompt), а валидирует то,
    что пришло — сразу видно, если модель что-то упустила или исказила тип."""
    corrected_transcript: str = Field(description="Транскрипт с исправленными ошибками ASR и реальными именами спикеров вместо 'Канал N'")
    meeting_title: str
    domain: str = Field(description="Предметная сфера встречи, например: строительство, IT, продажи")
    summary: str = Field(description="4-6 предложений деловой прозы")
    decisions: List[str] = Field(default_factory=list, description="Только реально принятые решения, дословно из транскрипта")
    action_items: List[ActionItem] = Field(default_factory=list)
    open_questions: List[str] = Field(default_factory=list, description="Вопросы, которые обсуждали, но не решили")

    @field_validator("meeting_title", mode="before")
    @classmethod
    def _fallback_title(cls, v):
        """Промпт просит модель никогда не оставлять meeting_title пустым (см.
        build_protocol_prompt, п.4), но на коротких/малоинформативных встречах
        модель иногда всё же возвращает null, применяя к заголовку правило
        "не выдумывай" — программная страховка на этот случай, а не только
        промпт-инструкция."""
        return v if v else "Встреча без определённой темы"


class ChunkOutput(BaseModel):
    """Ответ на исправление одного чанка — только текст, без полей протокола
    (decisions/action_items и т.п. считаем один раз по всему тексту сразу)."""
    corrected_transcript: str = Field(description="Исправленный текст этого куска транскрипта")


In [ ]:
CHUNK_SIZE_CHARS = 15000  # запас под max_output_tokens=8000: чем длиннее вход, тем длиннее нужный ответ

class ChunkBoundaries(BaseModel):
    """Ответ на разметку границ чанков — только номера строк, компактный
    вывод, не упирается в max_output_tokens даже на длинном транскрипте."""
    boundaries: List[int] = Field(description="Номера строк (с 1), после которых логично разрезать текст")


def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE_CHARS) -> List[str]:
    """Режет транскрипт на куски по смысловым границам (смена темы, конец
    обсуждения вопроса) — просим LLM разметить, где резать, а не режем
    вслепую по фиксированному числу символов. Модель возвращает только номера
    строк, это короткий ответ, поэтому не упирается в лимит даже на длинном
    входе. Если разметка не удалась — откатываемся на резку по границам реплик."""
    lines = text.split("\n")

    numbered = "\n".join(f"{i + 1}: {line}" for i, line in enumerate(lines))
    system_prompt = f"""Тебе дан транскрипт встречи, пронумерованный по строкам (одна строка —
одна реплика). Раздели его на куски примерно по {chunk_size} символов каждый,
разрезая по смысловым границам — конец обсуждения темы, смена вопроса, а не
посередине разговора. Верни только номера строк, ПОСЛЕ которых нужно резать.

Верни ТОЛЬКО корректный JSON без markdown-обёртки:
{{"boundaries": [номер_строки, номер_строки, ...]}}"""

    try:
        response = gpt_client.responses.create(
            model=MODEL_URI,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": numbered},
            ],
            temperature=0.1,
            max_output_tokens=1000,  # только список чисел, ответ короткий
        )
        candidate = response.output_text.strip()
        if candidate.startswith("```"):
            candidate = candidate.split("```")[1]
            candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
        boundaries = sorted(set(ChunkBoundaries.model_validate_json(candidate).boundaries))
        boundaries = [b for b in boundaries if 0 < b < len(lines)]
    except Exception as e:
        print(f"⚠️  Не удалось разметить границы чанков умно ({e}), режем по размеру")
        boundaries = []

    if not boundaries:
        return _split_by_size(lines, chunk_size)

    # LLM иногда режет слишком часто (на каждую смену темы, а не раз в
    # ~chunk_size) — отбрасываем границы, которые давали бы чанк меньше
    # половины целевого размера, иначе чанков будет в разы больше нужного
    # и каждый лишний чанк — это лишний платный вызов.
    min_chunk_len = chunk_size // 2
    filtered_boundaries: List[int] = []
    last_cut = 0
    for b in boundaries:
        piece_len = sum(len(l) + 1 for l in lines[last_cut:b])
        if piece_len >= min_chunk_len:
            filtered_boundaries.append(b)
            last_cut = b

    if not filtered_boundaries:
        return _split_by_size(lines, chunk_size)

    chunks: List[str] = []
    start = 0
    for b in filtered_boundaries:
        chunks.append("\n".join(lines[start:b]))
        start = b
    chunks.append("\n".join(lines[start:]))
    return [c for c in chunks if c.strip()]


def _split_by_size(lines: List[str], chunk_size: int) -> List[str]:
    """Запасной вариант: режет по границам реплик без учёта смысла —
    используется, если LLM-разметка границ не удалась."""
    chunks: List[str] = []
    current: List[str] = []
    current_len = 0
    for line in lines:
        if current and current_len + len(line) > chunk_size:
            chunks.append("\n".join(current))
            current, current_len = [], 0
        current.append(line)
        current_len += len(line) + 1
    if current:
        chunks.append("\n".join(current))
    return chunks


def build_chunk_prompt() -> str:
    """Промпт для одного чанка — только коррекция ASR, без протокола
    (короче, чем полный промпт, поэтому и ответ короче). Домен встречи здесь
    не нужен: он определяется моделью только в финальном протоколе, чанки
    исправляются нейтрально."""
    return """Ты — редактор транскриптов деловых встреч. Тебе дан кусок
транскрипта, полученного автоматическим распознаванием речи (ASR), с грубой
разметкой по каналам записи ([Канал 0], [Канал 1], ...). Возможно, это середина
разговора — в начале дан контекст конца предыдущего куска для связности.

ЗАДАЧА:
1. Разметь реплики по спикерам так же, как в примере контекста (если он дан) —
   сохраняй те же номера/имена для тех же людей.
2. Исправь искажения ASR: разорванные слова, аббревиатуры и марки, произнесённые
   словами ("м триста пятьдесят" → "М350"). НИКОГДА не меняй сами числа, даты и суммы.
3. НЕ добавляй и НЕ удаляй реплики, только исправляй текст внутри них.

Верни ТОЛЬКО корректный JSON без markdown-обёртки:
{"corrected_transcript": "исправленный текст этого куска"}"""


def refine_transcript_chunk(chunk: str, context_tail: Optional[str] = None) -> str:
    """Исправляет один чанк транскрипта отдельным вызовом LLM."""
    system_prompt = build_chunk_prompt()
    user_content = chunk if not context_tail else f"Конец предыдущего куска (для контекста):\n{context_tail}\n\nТекущий кусок:\n{chunk}"

    response = gpt_client.responses.create(
        model=MODEL_URI,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        temperature=0.2,
        max_output_tokens=8000,
    )
    candidate = response.output_text.strip()
    if candidate.startswith("```"):
        candidate = candidate.split("```")[1]
        candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
    return ChunkOutput.model_validate_json(candidate).corrected_transcript


In [ ]:
def build_protocol_prompt(project_team: List[Dict[str, str]]) -> str:
    """В промпт кладём готовый ПРИМЕР ответа с реальными значениями, а не дамп
    JSON Schema (properties/type/description) — модель на практике путает такую
    схему с данными и подставляет метаданные полей вместо самих значений."""
    team_block = "\n".join(f"- {m['name']} ({m.get('role', 'без роли')})" for m in project_team) \
        or "Состав участников неизвестен — используй 'Спикер 1', 'Спикер 2' и т.д."

    example = {
        "corrected_transcript": "Иванов Алексей: Коллеги, начнём с фундамента...",
        "meeting_title": "Планёрка по объекту на Садовой",
        "domain": "строительство",
        "summary": "4-6 предложений деловой прозы.",
        "decisions": ["решение дословно из транскрипта"],
        "action_items": [{"owner": "Иванов Алексей", "task": "конкретная задача", "deadline": "2026-08-15 или null"}],
        "open_questions": ["вопрос, который обсуждали, но не решили"],
    }
    example_json = json.dumps(example, ensure_ascii=False, indent=2)

    return f"""Ты — секретарь деловых встреч. Тебе дан транскрипт, полученный автоматическим
распознаванием речи (ASR), с грубой разметкой по каналам записи ([Канал 0], [Канал 1], ...).

УЧАСТНИКИ ВСТРЕЧИ (известный состав):
{team_block}

ЗАДАЧИ:
1. Разметь реплики по реальным участникам. Разметка по каналам — подсказка о числе
   говорящих, а не факт: если она выглядит нарезанной через каждые несколько слов,
   перегруппируй реплики по смыслу (вопрос → ответ, обращение по имени, смена темы).
   Если имя не удаётся определить уверенно — оставь метку "Спикер N".
2. Исправь искажения ASR: разорванные слова, аббревиатуры и марки, произнесённые
   словами ("м триста пятьдесят" → "М350"). Пунктуация и заглавные буквы — по смыслу.
   НИКОГДА не меняй сами числа, даты и суммы — только форму записи.
3. Составь протокол: реальные решения ("решили", "утвердили"), задачи с ответственным
   и дедлайном (если не назван — "Команда" и null), открытые вопросы, которые
   обсуждали, но не решили.
4. "meeting_title" — ВСЕГДА строка, никогда null: если из транскрипта не ясна
   конкретная тема встречи, используй общее описание по домену и участникам
   (например "Планёрка по текущим задачам"), а не оставляй поле пустым.

НЕ ВЫДУМЫВАЙ факты. Если решений или задач нет в тексте — верни пустой список,
а не пример "для порядка". Это правило не относится к meeting_title (см. п.4) —
заголовок обязателен как ярлык, даже когда тема расплывчата.

Верни ТОЛЬКО корректный JSON без markdown-обёртки, СТРОГО в этом формате
(ниже — форма ответа с примерами значений, не переписывай примеры дословно):
{example_json}"""


In [ ]:
def parse_protocol_json(raw_text: str) -> ProtocolOutput:
    """Модель иногда оборачивает JSON в ```json ... ``` — снимаем обёртку перед парсингом."""
    candidate = raw_text.strip()
    if candidate.startswith("```"):
        candidate = candidate.split("```")[1]
        candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
    return ProtocolOutput.model_validate_json(candidate)


def refine_long_transcript(raw_transcript: str) -> str:
    """Для длинных встреч: режет транскрипт на чанки, исправляет каждый
    отдельным вызовом (без протокола — так каждый ответ короче и не упирается
    в max_output_tokens), передавая хвост предыдущего чанка как контекст для
    связности между кусками, и склеивает результат обратно в один текст."""
    chunks = split_into_chunks(raw_transcript)
    print(f"Транскрипт длинный — режем на {len(chunks)} чанков")

    refined_chunks: List[str] = []
    context_tail: Optional[str] = None
    for i, chunk in enumerate(chunks, start=1):
        refined = refine_transcript_chunk(chunk, context_tail)
        refined_chunks.append(refined)
        context_tail = refined[-500:]  # хвост для связности со следующим чанком
        print(f"  чанк {i}/{len(chunks)} исправлен")

    return "\n".join(refined_chunks)


def extract_meeting_protocol(raw_transcript: str, project_team: List[Dict[str, str]]) -> ProtocolOutput:
    """Транскрипт + состав команды на входе, исправленный текст с реальными
    именами и готовый протокол на выходе. Короткий текст обрабатывается одним
    вызовом; длинный сначала чанкуется и исправляется по кускам (см.
    refine_long_transcript), а протокол собирается уже по всему тексту сразу."""
    if len(raw_transcript) > CHUNK_SIZE_CHARS:
        raw_transcript = refine_long_transcript(raw_transcript)

    system_prompt = build_protocol_prompt(project_team)

    response = gpt_client.responses.create(
        model=MODEL_URI,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Транскрипт:\n{raw_transcript[:20000]}"},
        ],
        temperature=0.2,
        max_output_tokens=8000,  # ответ включает исправленный транскрипт целиком
    )
    return parse_protocol_json(response.output_text)


# 7. Сборка пайплайна


In [ ]:
def run_meeting_pipeline(audio_local_path: str, project_team: List[Dict[str, str]]) -> Tuple[ProtocolOutput, str]:
    """Возвращает (протокол, meeting_id) — id нужен, чтобы сохранить результат
    прогона в файл с тем же именем, что и аудио в Object Storage."""
    meeting_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    audio_uri = upload_audio(audio_local_path, f"audio/{meeting_id}.wav")

    raw_chunks = recognize_meeting_audio(audio_uri)
    raw_text = postprocess_transcript(raw_chunks)
    print(f"Транскрипт: {len(raw_text)} символов")

    protocol = extract_meeting_protocol(raw_text, project_team)
    return protocol, meeting_id


def save_transcript_to_txt(protocol: ProtocolOutput, meeting_id: str) -> str:
    """Сохраняет только исправленный транскрипт — отдельным файлом от протокола,
    рядом с ноутбуком (текущая рабочая директория)."""
    out_path = f"transcript_{meeting_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(protocol.corrected_transcript + "\n")
    print(f"Сохранено: {out_path}")
    return out_path


def save_protocol_to_txt(protocol: ProtocolOutput, meeting_id: str) -> str:
    """Сохраняет только протокол (JSON) — отдельным файлом от транскрипта."""
    out_path = f"protocol_{meeting_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(protocol.model_dump(), ensure_ascii=False, indent=2) + "\n")
    print(f"Сохранено: {out_path}")
    return out_path


# 8. Тестирование на реальном аудиофайле

Замените `AUDIO_LOCAL_PATH` на свой WAV-файл и впишите реальных участников встречи, чтобы увидеть диаризацию по именам.


In [ ]:
AUDIO_LOCAL_PATH = "your_meeting_recording.wav"  # <-- впишите путь к своему аудиофайлу (WAV/OggOpus/MP3)

project_team = [
    {"name": "Иванов Алексей", "role": "прораб"},
    {"name": "Петрова Мария", "role": "заказчик"},
    {"name": "Сидоров Игорь", "role": "инженер ПТО"},
]


In [ ]:
protocol, meeting_id = run_meeting_pipeline(AUDIO_LOCAL_PATH, project_team)


In [ ]:
print("=" * 80)
print("ИСПРАВЛЕННЫЙ ТРАНСКРИПТ")
print("=" * 80)
print(protocol.corrected_transcript)

save_transcript_to_txt(protocol, meeting_id)


In [ ]:
print("=" * 80)
print("ПРОТОКОЛ")
print("=" * 80)
print(json.dumps(protocol.model_dump(), ensure_ascii=False, indent=2))

save_protocol_to_txt(protocol, meeting_id)


# 9. Результаты и анализ

## Что мы достигли

Полный цикл «аудио → протокол» на трёх сервисах Yandex Cloud:
- аудио загружено в Object Storage;
- SpeechKit вернул транскрипт с разметкой по каналам;
- для коротких встреч YandexGPT одним вызовом расставил реальные имена участников, исправил ошибки ASR и собрал протокол; для длинных — сначала разбил транскрипт на смысловые чанки и исправил их по отдельности, а протокол собрал уже по всему тексту;
- транскрипт и протокол сохранены отдельными файлами (`transcript_<id>.txt`, `protocol_<id>.txt`).

## Как это работает

Диаризация здесь не полагается на акустическую разметку SpeechKit как на последнюю инстанцию: модель получает и разметку по каналам, и состав команды проекта, и сама решает, где реально сменился говорящий — по прямым обращениям, характерной лексике роли, структуре диалога (вопрос → ответ). Программная страховка (`field_validator` в `ProtocolOutput`) подхватывает случаи, когда модель всё же нарушает инструкцию промпта — например, возвращает `null` вместо обязательного заголовка встречи.

Длинные встречи не помещаются в один ответ LLM (см. раздел 6) — `refine_long_transcript` режет транскрипт на чанки, но не вслепую по числу символов, а по смысловым границам, которые тоже находит LLM: отдельным лёгким вызовом с коротким выводом (только номера строк), не упирающимся в лимит. Каждый чанк исправляется по отдельности, с хвостом предыдущего куска как контекстом для связности, и результаты склеиваются обратно в один текст.

## Ключевые параметры

- `temperature=0.2` — консервативная настройка для задачи, где важна точность (коррекция чисел, дат), а не творческое разнообразие.
- `max_output_tokens=8000` — ответ включает исправленный транскрипт целиком, поэтому лимит выше типового.
- `CHUNK_SIZE_CHARS=15000` — порог, после которого транскрипт режется на чанки перед коррекцией.
- `GPT_MODEL=yandexgpt-5-lite` — выбрана по цене/скорости для учебного примера; для более сложных доменов сравните с `yandexgpt-5-pro` на своих записях.

## Ограничения учебной версии

- Финальный вызов, который собирает протокол, всё равно ограничен `max_output_tokens=8000` и обрезает вход до 20 000 символов (`raw_transcript[:20000]`) — на очень длинных встречах (многочасовых) даже после чанкинга коррекции текст для протокола может не поместиться целиком. В проде этот шаг тоже стоит развести на отдельные вызовы (например, промежуточные сводки по частям встречи + один финальный вызов на консолидацию).
- Модель может пропустить реплику при большом объёме входного текста — для прод-сценария стоит добавить проверку целостности (построчную нумерацию реплик и сверку, что каждая вернулась) и программный валидатор чисел/дат вместо расчёта только на промпт-инструкцию.
- Диаризация иногда дробит одну связную мысль говорящего на несколько коротких реплик разных «спикеров» — особенно на репликах-подтверждениях («да», «понял», «хорошо»). Если это критично для вашего сценария, стоит усилить инструкцию промпта или добавить пост-обработку, которая склеивает подряд идущие короткие реплики.
- Оценивайте качество коррекции на своих записях: возьмите 10-20 реальных фрагментов, аннотируйте эталонный текст руками и сравните с результатом модели, прежде чем фиксировать выбор модели в проде.

## Куда двигаться дальше

- Разнести старт распознавания и опрос операции по отдельным вызовам (Cloud Functions + очередь), если сервис асинхронный и без общего процесса ожидания.
- Добавить QA-проход, который проверяет пункты протокола на соответствие транскрипту.


# 10. Полезные ссылки

- [SpeechKit STT v3, асинхронное распознавание](https://aistudio.yandex.ru/docs/ru/speechkit/stt/api/transcribation-api-v3)
- [Поддерживаемые форматы аудио](https://aistudio.yandex.ru/docs/ru/speechkit/formats)
- [Доступные генеративные модели](https://aistudio.yandex.ru/docs/ru/ai-studio/concepts/generation/models)
- [Object Storage (S3-совместимое API)](https://yandex.cloud/ru/docs/storage/s3/)
